# Kepler CSV preprocessing experiment
In Colab select a CPU runtime and Run all. Upload exoTrain.csv from .transiteye-workspace/data/kaggle_kepler_labelled/. This is an exploratory fit to the supplied Kaggle labels. The CSV has no source IDs for checking star-level partitions. The development model in the companion notebook uses FITS-derived TESS candidates.

In [ ]:
import hashlib, json, shutil, tempfile
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn, joblib
from sklearn.ensemble import RandomForestClassifier
from google.colab import files
path = Path('exoTrain.csv')
if not path.is_file():
    files.upload()
with path.open('rb') as stream:
    digest = hashlib.sha256()
    for block in iter(lambda: stream.read(1048576), b''):
        digest.update(block)
assert digest.hexdigest() == '6f04167636d2fcdce44af6fd5cbef359d8af033ce8457d8c8e917fdb822e044c'
df = pd.read_csv(path, dtype=np.float32)
assert list(df.columns) == ['LABEL'] + [f'FLUX.{i}' for i in range(1,3198)]
y = df.pop('LABEL').to_numpy().astype(int)
assert len(y) == 5087 and set(y) == {1,2}
X = df.to_numpy()
assert np.isfinite(X).all()
print(pd.Series(y).value_counts())

## Fit two feature sets
The baseline uses quantiles of each median-centered curve scaled by its median absolute deviation. The second forest also uses Fourier power in eight frequency bands. Frequencies are in cycles per sample. The displayed importances describe each fit. These data cannot establish a gain in accuracy on unseen stars.

In [ ]:
def feature_sets(curves):
    centered = curves - np.median(curves, axis=1, keepdims=True)
    mad = np.median(abs(centered), axis=1, keepdims=True)
    z = centered / np.where(mad > 0, mad, 1)
    quantiles = np.quantile(z, [.001,.01,.05,.25,.75,.95,.99,.999], axis=1).T
    power = abs(np.fft.rfft(z, axis=1)[:,1:])**2
    bands = np.stack([p.sum(axis=1) for p in np.array_split(power,8,axis=1)], axis=1)
    return {'quantiles':quantiles, 'quantiles_fft': np.column_stack([quantiles,np.log1p(bands)])}

out = Path(tempfile.mkdtemp(prefix='kepler-experiment-'))
for name, features in feature_sets(X).items():
    forest = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=2,
                                   random_state=42, class_weight='balanced_subsample')
    forest.fit(features,y)
    joblib.dump(forest,out/(name+'.joblib'))
    print(name, np.round(forest.feature_importances_,3))
(out/'run.json').write_text(json.dumps(dict(input_sha256=digest.hexdigest(), seed=42,
    classes=forest.classes_.tolist(), numpy=np.__version__, sklearn=sklearn.__version__,
    usage='Exploratory fit on the supplied training rows. Generalization unmeasured.'),indent=2))
files.download(shutil.make_archive(str(out),'zip',out))